[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AllInVaders/aistudio-full-course/blob/main/notebooks/02_Prompts_Structured_Outputs_Gemini_Image_and_Omni_Video.ipynb)

# Module 02: System Instructions, Thinking Levels, Structured Outputs, Nano Banana & Gemini Omni Flash
### Módulo 02: Instrucciones de Sistema, Niveles de Razonamiento, Salidas Estructuradas, Nano Banana y Gemini Omni Flash

**English Overview**: Build **Stage 1 of the Flagship Project (AI Product Studio Creative Engine)**. You will generate guaranteed Pydantic JSON launch kits with `gemini-3.8-flash`, render photorealistic studio hero shots with **Nano Banana** (`gemini-3.1-flash-image`), generate cinematic promo clips with **Gemini Omni Flash** (`gemini-omni-1.1-flash`), and refine both conversationally by chaining `previous_interaction_id`.

**Resumen en Español**: Construye la **Etapa 1 del Proyecto Insignia (Motor Creativo de Estudio de Productos IA)**. Generarás kits de lanzamiento JSON validados con Pydantic usando `gemini-3.8-flash`, fotografías de estudio fotorrealistas con **Nano Banana** (`gemini-3.1-flash-image`), clips promocionales cinematográficos con **Gemini Omni Flash** (`gemini-omni-1.1-flash`), y refinarás ambos de forma conversacional encadenando `previous_interaction_id`.

Referencias: [salidas estructuradas](https://ai.google.dev/gemini-api/docs/structured-output) · [generación de imágenes](https://ai.google.dev/gemini-api/docs/image-generation) · [generación de video](https://ai.google.dev/gemini-api/docs/omni)

In [ ]:
%pip install -q -U google-genai pydantic pillow

In [ ]:
import base64
import json
from io import BytesIO
from typing import List

from google import genai
from pydantic import BaseModel, Field

class ChannelAdCopy(BaseModel):
    channel: str
    headline_en: str
    headline_es: str
    body_copy_en: str
    body_copy_es: str

class ProductLaunchKit(BaseModel):
    product_name: str
    tagline_en: str
    tagline_es: str
    hero_image_prompt: str = Field(description='Detailed studio photography prompt: lighting, lens, materials, palette.')
    promo_video_prompt: str = Field(description='Cinematic 5-second motion prompt: camera movement, subject action, lighting.')
    ad_campaigns: List[ChannelAdCopy]

client = genai.Client()

TEXT_MODEL = 'gemini-3.8-flash'
IMAGE_MODEL = 'gemini-3.1-flash-image'      # Nano Banana 2
IMAGE_MODEL_PRO = 'gemini-3-pro-image'      # Nano Banana Pro: 4K + precise text
VIDEO_MODEL = 'gemini-omni-1.1-flash'       # Gemini Omni Flash

## 1. Structured Bilingual Launch Kit / Kit de Lanzamiento Bilingüe

`response_format` pins the reply to our Pydantic JSON schema, so
`ProductLaunchKit.model_validate_json` can never receive malformed output.
`thinking_level` trades latency for reasoning depth — `'medium'` is a good
default for creative-but-grounded work.

In [ ]:
concept = 'AeroBrew Nano: Pocket-sized ultrasonic cold-brew espresso maker for frequent flyers'

interaction = client.interactions.create(
    model=TEXT_MODEL,
    input=f'Create a complete bilingual launch kit for: {concept}',
    system_instruction='You are an Executive Creative Director at an AI Product Studio.',
    response_format={
        'type': 'text',
        'mime_type': 'application/json',
        'schema': ProductLaunchKit.model_json_schema(),
    },
    generation_config={'temperature': 0.4, 'thinking_level': 'medium'},
)

launch_kit = ProductLaunchKit.model_validate_json(interaction.output_text)
print(json.dumps(launch_kit.model_dump(), indent=2, ensure_ascii=False))

## 2. Studio Hero Shot with Nano Banana / Fotografía Hero con Nano Banana

Image generation runs through the same `interactions.create` call — only the
model and the `response_format` change. The PNG comes back base64-encoded on
`interaction.output_image.data`.

Swap `IMAGE_MODEL` for `IMAGE_MODEL_PRO` when you need 4K output or crisp text
rendered inside the image; the request shape is identical.

Supported aspect ratios: `1:1`, `2:3`, `3:4`, `4:3`, `4:5`, `5:4`, `9:16`, `16:9`, `21:9`.

> Every generated image carries a [SynthID watermark](https://ai.google.dev/responsible/docs/safeguards/synthid).

In [ ]:
from PIL import Image

image_interaction = client.interactions.create(
    model=IMAGE_MODEL,
    input=launch_kit.hero_image_prompt,
    response_format={
        'type': 'image',
        'mime_type': 'image/png',
        'aspect_ratio': '16:9',
        'image_size': '2K',
    },
)

hero_bytes = base64.b64decode(image_interaction.output_image.data)
display(Image.open(BytesIO(hero_bytes)))
print('Interaction ID:', image_interaction.id)

## 3. Conversational Image Editing / Edición Conversacional

Passing `previous_interaction_id` edits the render you already have instead of
rolling the dice on a brand new one, so the product, framing, and lighting stay
consistent across turns.

Remember: `tools`, `system_instruction`, and `generation_config` are
interaction-scoped, so re-specify them on every turn.

In [ ]:
edit_interaction = client.interactions.create(
    model=IMAGE_MODEL,
    input='Keep the product and composition identical, but swap the background for brushed concrete and warm the key light by 300K.',
    previous_interaction_id=image_interaction.id,
    response_format={
        'type': 'image',
        'mime_type': 'image/png',
        'aspect_ratio': '16:9',
        'image_size': '2K',
    },
)

edited_bytes = base64.b64decode(edit_interaction.output_image.data)
display(Image.open(BytesIO(edited_bytes)))

> **Why we did not pass `store=False` here.**
> Conversational editing depends on server-side interaction history, and
> `store=False` disables `previous_interaction_id` (it is also incompatible with
> `background=true`). If you need the privacy of `store=False`, you give up
> in-place editing and must re-describe the full scene on every render.
> Retention defaults to 55 days on the Paid Tier and 1 day on the Free Tier.
> See https://ai.google.dev/gemini-api/docs/interactions-overview

## 4. Cinematic Promo Clip with Gemini Omni Flash / Clip Promocional con Gemini Omni Flash

Gemini Omni Flash handles video generation **and** editing, keyframe
interpolation, scene extension, and natively synchronized audio.

The MP4 is returned inline as base64 on `interaction.output_video.data`. There
is no long-running operation to poll. Supported aspect ratios are `16:9`
(default) and `9:16`.

> **Legacy migration note.** Veo and `client.models.generate_videos()` are
> retired. See https://ai.google.dev/gemini-api/docs/migrate-to-interactions

In [ ]:
from IPython.display import Video

video_interaction = client.interactions.create(
    model=VIDEO_MODEL,
    input=launch_kit.promo_video_prompt,
    response_format={'type': 'video', 'aspect_ratio': '16:9'},
)

with open('promo_teaser.mp4', 'wb') as f:
    f.write(base64.b64decode(video_interaction.output_video.data))

print('Saved promo_teaser.mp4')
Video('promo_teaser.mp4', embed=True, width=640)

### Conversational video editing / Edición conversacional de video

The same chaining trick works for video. Ask for a change in plain language and
Gemini Omni Flash keeps the rest of the shot consistent.

In [ ]:
revised_video = client.interactions.create(
    model=VIDEO_MODEL,
    input='Extend the shot by two seconds and end on a slow push-in to the logo.',
    previous_interaction_id=video_interaction.id,
    response_format={'type': 'video', 'aspect_ratio': '16:9'},
)

with open('promo_teaser_v2.mp4', 'wb') as f:
    f.write(base64.b64decode(revised_video.output_video.data))

print('Saved promo_teaser_v2.mp4')